In [0]:
%sql
-- Owner: Cath
-- Name: 00 - Gold Setup
-- Purpose: Create the Gold (Mart) schema for dimensional modeling.
-- Grain: N/A - schema creation only.

USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS instacart_gold;
USE SCHEMA instacart_gold;

## Part 1: Dimensions

Dimension tables for the Instacart mart - denormalized for easier querying.

In [0]:
%sql
-- Owner: Cath
-- Name: 15_gold_dim_product.sql
-- Purpose: Build a product dimension with denormalized aisle and department names.
-- Grain: One row per product, uniquely identified by product_id.

CREATE OR REPLACE TABLE gold_dim_product AS
SELECT
    p.product_id,
    p.product_name,
    p.aisle_id,
    a.aisle AS aisle_name,
    p.department_id,
    d.department AS department_name
FROM instacart_silver.products_clean p
LEFT JOIN instacart_silver.aisles_clean a ON p.aisle_id = a.aisle_id
LEFT JOIN instacart_silver.departments_clean d ON p.department_id = d.department_id;

DESCRIBE TABLE gold_dim_product;

In [0]:
%sql
-- Owner: Cath
-- Name: 16_gold_dim_order.sql
-- Purpose: Build an order dimension with customer and time attributes.
-- Grain: One row per order, uniquely identified by order_id.

CREATE OR REPLACE TABLE gold_dim_order AS
SELECT
    order_id,
    user_id,
    order_number,
    order_dow,
    CASE order_dow
        WHEN 0 THEN 'Sunday'
        WHEN 1 THEN 'Monday'
        WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday'
        WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'
        WHEN 6 THEN 'Saturday'
    END AS order_day_name,
    order_hour_of_day,
    days_since_prior_order,
    CASE
        WHEN days_since_prior_order IS NULL THEN 'First Order'
        WHEN days_since_prior_order <= 7 THEN 'Within 1 Week'
        WHEN days_since_prior_order <= 14 THEN '1-2 Weeks'
        WHEN days_since_prior_order <= 30 THEN '2-4 Weeks'
        ELSE 'Over 30 Days'
    END AS order_frequency_category
FROM instacart_silver.orders_clean;

DESCRIBE TABLE gold_dim_order;

SELECT * FROM gold_dim_order;

## Part 2: Fact Table

The fact table contains the transactional grain - one row per product in each order.

In [0]:
%sql
-- Owner: Cath
-- Name: 17_gold_fact_order_product.sql
-- Purpose: Build the fact table linking orders and products with behavioral metrics.
-- Grain: One row per product line in one order, uniquely identified by (order_id, product_id).

CREATE OR REPLACE TABLE gold_fact_order_product AS
SELECT
    op.order_id,
    op.product_id,
    op.add_to_cart_order,
    op.reordered,
    o.user_id,
    o.order_number,
    o.order_dow,
    o.order_hour_of_day,
    p.aisle_id,
    p.department_id
FROM instacart_silver.order_products_clean op
INNER JOIN instacart_silver.orders_clean o ON op.order_id = o.order_id
INNER JOIN instacart_silver.products_clean p ON op.product_id = p.product_id;

DESCRIBE TABLE gold_fact_order_product;

In [0]:
%sql
-- Owner: Cath
-- Name: 18_validate_gold_pre_constraints.sql
-- Purpose: Validate dimension tables before building the fact table.
-- Grain: One validation summary row per dimension table.

WITH validation AS (

    SELECT
        'gold_dim_product' AS table_name,
        COUNT(*) AS row_count,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_key_rows,
        (SELECT COUNT(*) FROM (
            SELECT product_id FROM gold_dim_product
            WHERE product_id IS NOT NULL GROUP BY product_id HAVING COUNT(*) > 1
        )) AS duplicate_keys,
        SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS required_field_issues
    FROM gold_dim_product

    UNION ALL

    SELECT
        'gold_dim_order',
        COUNT(*),
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT order_id FROM gold_dim_order
            WHERE order_id IS NOT NULL GROUP BY order_id HAVING COUNT(*) > 1
        )),
        SUM(CASE WHEN user_id IS NULL OR order_number IS NULL THEN 1 ELSE 0 END)
    FROM gold_dim_order

)
SELECT
    table_name,
    row_count,
    null_key_rows,
    duplicate_keys,
    required_field_issues,
    CASE
        WHEN null_key_rows > 0 OR duplicate_keys > 0 OR required_field_issues > 0
        THEN 'REVIEW'
        ELSE 'PASS'
    END AS status
FROM validation
ORDER BY table_name;

In [0]:
%sql
-- Owner: Cath
-- Name: 19_gold_constraints.sql
-- Purpose: Validate PK/FK constraints across all gold tables.
-- Grain: One validation summary row per constraint check.

WITH constraint_checks AS (

    -- Check 1: gold_dim_product PK uniqueness
    SELECT
        'PK: gold_dim_product.product_id' AS constraint_name,
        'Primary Key Uniqueness' AS constraint_type,
        (SELECT COUNT(*) FROM (
            SELECT product_id FROM gold_dim_product
            WHERE product_id IS NOT NULL
            GROUP BY product_id HAVING COUNT(*) > 1
        )) AS violations

    UNION ALL

    -- Check 2: gold_dim_order PK uniqueness
    SELECT
        'PK: gold_dim_order.order_id',
        'Primary Key Uniqueness',
        (SELECT COUNT(*) FROM (
            SELECT order_id FROM gold_dim_order
            WHERE order_id IS NOT NULL
            GROUP BY order_id HAVING COUNT(*) > 1
        ))

    UNION ALL

    -- Check 3: gold_fact_order_product composite PK uniqueness
    SELECT
        'PK: gold_fact_order_product (order_id, product_id)',
        'Composite Key Uniqueness',
        (SELECT COUNT(*) FROM (
            SELECT order_id, product_id FROM gold_fact_order_product
            WHERE order_id IS NOT NULL AND product_id IS NOT NULL
            GROUP BY order_id, product_id HAVING COUNT(*) > 1
        ))

    UNION ALL

    -- Check 4: gold_dim_product PK null check
    SELECT
        'PK: gold_dim_product.product_id NOT NULL',
        'Primary Key Null Check',
        (SELECT COUNT(*) FROM gold_dim_product WHERE product_id IS NULL)

    UNION ALL

    -- Check 5: gold_dim_order PK null check
    SELECT
        'PK: gold_dim_order.order_id NOT NULL',
        'Primary Key Null Check',
        (SELECT COUNT(*) FROM gold_dim_order WHERE order_id IS NULL)

    UNION ALL

    -- Check 6: gold_fact_order_product composite PK null check
    SELECT
        'PK: gold_fact_order_product (order_id, product_id) NOT NULL',
        'Composite Key Null Check',
        (SELECT COUNT(*) FROM gold_fact_order_product
         WHERE order_id IS NULL OR product_id IS NULL)

    UNION ALL

    -- Check 7: FK gold_fact_order_product.order_id → gold_dim_order.order_id
    SELECT
        'FK: gold_fact_order_product.order_id → gold_dim_order.order_id',
        'Foreign Key Integrity',
        (SELECT COUNT(*) FROM gold_fact_order_product f
         LEFT JOIN gold_dim_order d ON f.order_id = d.order_id
         WHERE f.order_id IS NOT NULL AND d.order_id IS NULL)

    UNION ALL

    -- Check 8: FK gold_fact_order_product.product_id → gold_dim_product.product_id
    SELECT
        'FK: gold_fact_order_product.product_id → gold_dim_product.product_id',
        'Foreign Key Integrity',
        (SELECT COUNT(*) FROM gold_fact_order_product f
         LEFT JOIN gold_dim_product d ON f.product_id = d.product_id
         WHERE f.product_id IS NOT NULL AND d.product_id IS NULL)

)
SELECT
    constraint_name,
    constraint_type,
    violations,
    CASE WHEN violations = 0 THEN 'PASS' ELSE 'REVIEW' END AS status
FROM constraint_checks
ORDER BY constraint_type, constraint_name;

## Part 2: Fact Table

The fact table contains the transactional grain - one row per product in each order.

In [0]:
%sql
-- Owner: Cath
-- Name: 17_gold_fact_order_product.sql
-- Purpose: Build the fact table linking orders and products with behavioral metrics.
-- Grain: One row per product line in one order, uniquely identified by (order_id, product_id).

CREATE OR REPLACE TABLE gold_fact_order_product AS
SELECT
    op.order_id, -- connects to dim_order table
    op.product_id, -- connects to dim_product table
    op.add_to_cart_order,
    op.reordered
FROM instacart_silver.order_products_clean op;

DESCRIBE TABLE gold_fact_order_product;

## Part 3: Validation

Validate the gold layer for data quality, referential integrity, and constraints.

In [0]:
%sql
-- Owner: Cath
-- Name: 20_validate_gold_final.sql
-- Purpose: Comprehensive gold validation - keys, integrity, silver-to-gold reconciliation, and measures.
-- Grain: Two result sets - (1) table-level validation, (2) measure reconciliation.

-- PART 1: Table-level validation (keys, row counts, referential integrity)
WITH validation AS (

    SELECT
        'gold_dim_product' AS table_name,
        COUNT(*) AS row_count,
        (SELECT COUNT(*) FROM instacart_silver.products_clean) AS source_row_count,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_key_rows,
        (SELECT COUNT(*) FROM (
            SELECT product_id FROM gold_dim_product
            WHERE product_id IS NOT NULL GROUP BY product_id HAVING COUNT(*) > 1
        )) AS duplicate_keys,
        SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS required_field_issues,
        0 AS unmatched_fk_rows
    FROM gold_dim_product

    UNION ALL

    SELECT
        'gold_dim_order',
        COUNT(*),
        (SELECT COUNT(*) FROM instacart_silver.orders_clean),
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT order_id FROM gold_dim_order
            WHERE order_id IS NOT NULL GROUP BY order_id HAVING COUNT(*) > 1
        )),
        SUM(CASE WHEN user_id IS NULL OR order_number IS NULL THEN 1 ELSE 0 END),
        0
    FROM gold_dim_order

    UNION ALL

    SELECT
        'gold_fact_order_product',
        COUNT(*),
        (SELECT COUNT(*) FROM instacart_silver.order_products_clean),
        SUM(CASE WHEN order_id IS NULL OR product_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT order_id, product_id FROM gold_fact_order_product
            WHERE order_id IS NOT NULL AND product_id IS NOT NULL
            GROUP BY order_id, product_id HAVING COUNT(*) > 1
        )),
        0,
        -- Check referential integrity: fact → dimensions
        (SELECT COUNT(*) FROM gold_fact_order_product f
            LEFT JOIN gold_dim_order o ON f.order_id = o.order_id
            WHERE f.order_id IS NOT NULL AND o.order_id IS NULL)
         + (SELECT COUNT(*) FROM gold_fact_order_product f
            LEFT JOIN gold_dim_product p ON f.product_id = p.product_id
            WHERE f.product_id IS NOT NULL AND p.product_id IS NULL)
    FROM gold_fact_order_product

)
SELECT
    table_name,
    row_count,
    source_row_count,
    row_count - source_row_count AS row_difference,
    null_key_rows,
    duplicate_keys,
    required_field_issues,
    unmatched_fk_rows,
    CASE
        WHEN null_key_rows > 0 OR duplicate_keys > 0
          OR required_field_issues > 0 OR unmatched_fk_rows > 0
          OR row_count <> source_row_count
        THEN 'REVIEW'
        ELSE 'PASS'
    END AS status
FROM validation
ORDER BY table_name;

-- PART 2: Measure reconciliation (aggregate validation)
WITH silver_measures AS (
    SELECT
        COUNT(*) AS total_order_lines,
        COUNT(DISTINCT order_id) AS distinct_orders,
        COUNT(DISTINCT product_id) AS distinct_products,
        SUM(CASE WHEN reordered = TRUE THEN 1 ELSE 0 END) AS reordered_count,
        SUM(add_to_cart_order) AS total_cart_position_sum
    FROM instacart_silver.order_products_clean
),
gold_measures AS (
    SELECT
        COUNT(*) AS total_order_lines,
        COUNT(DISTINCT order_id) AS distinct_orders,
        COUNT(DISTINCT product_id) AS distinct_products,
        SUM(CASE WHEN reordered = TRUE THEN 1 ELSE 0 END) AS reordered_count,
        SUM(add_to_cart_order) AS total_cart_position_sum
    FROM gold_fact_order_product
)
SELECT
    'total_order_lines' AS measure,
    s.total_order_lines AS silver_value,
    g.total_order_lines AS gold_value,
    s.total_order_lines - g.total_order_lines AS difference,
    CASE WHEN s.total_order_lines = g.total_order_lines THEN 'PASS' ELSE 'REVIEW' END AS status
FROM silver_measures s, gold_measures g

UNION ALL

SELECT
    'distinct_orders',
    s.distinct_orders,
    g.distinct_orders,
    s.distinct_orders - g.distinct_orders,
    CASE WHEN s.distinct_orders = g.distinct_orders THEN 'PASS' ELSE 'REVIEW' END
FROM silver_measures s, gold_measures g

UNION ALL

SELECT
    'distinct_products',
    s.distinct_products,
    g.distinct_products,
    s.distinct_products - g.distinct_products,
    CASE WHEN s.distinct_products = g.distinct_products THEN 'PASS' ELSE 'REVIEW' END
FROM silver_measures s, gold_measures g

UNION ALL

SELECT
    'reordered_count',
    s.reordered_count,
    g.reordered_count,
    s.reordered_count - g.reordered_count,
    CASE WHEN s.reordered_count = g.reordered_count THEN 'PASS' ELSE 'REVIEW' END
FROM silver_measures s, gold_measures g

UNION ALL

SELECT
    'total_cart_position_sum',
    s.total_cart_position_sum,
    g.total_cart_position_sum,
    s.total_cart_position_sum - g.total_cart_position_sum,
    CASE WHEN s.total_cart_position_sum = g.total_cart_position_sum THEN 'PASS' ELSE 'REVIEW' END
FROM silver_measures s, gold_measures g;

## Gold Layer: Summary

| Table | Status | Notes |
|---|---|---|
| `gold_dim_product` | Built + validated | Denormalized product dimension with aisle and department names |
| `gold_dim_order` | Built + validated | Order dimension with time attributes and frequency categories |
| `gold_fact_order_product` | Built + validated | Fact table linking orders and products with metrics |

**Expected result:** `status = 'PASS'` on all validation checks.

**Star Schema Design:**
- **Fact:** `gold_fact_order_product` (33.8M rows) - transactional grain, one row per product in each order
- **Dimension:** `gold_dim_product` (49.7K rows) - product attributes with denormalized aisle/department
- **Dimension:** `gold_dim_order` (3.4M rows) - order attributes with time and frequency metrics

**Validation Coverage:**

*Query 18 (Pre-Constraints):*
* Dimension key uniqueness (no duplicate PKs)
* Null primary keys (must be 0)
* Required field completeness

*Query 19 (Constraints):*
* Primary key uniqueness (3 checks: product_id, order_id, composite key)
* Primary key null checks (3 checks)
* Foreign key integrity (2 checks: fact → dimensions)

*Query 20 (Final Validation) - Two result sets:*

Part 1 - Table-level validation:
* Silver vs Gold row reconciliation (row_count vs source_row_count)
* Dimension key uniqueness (no duplicate PKs)
* Fact grain uniqueness (no duplicate composite keys)
* Null foreign keys (must be 0)
* Unmatched relationships (fact → dimension integrity)

Part 2 - Measure reconciliation:
* Total order lines (silver vs gold)
* Distinct orders preserved
* Distinct products preserved  
* Reordered count accuracy
* Cart position sum accuracy

**Next stage:** Dashboard - build analytics and visualizations from `workspace.instacart_gold`.